# Hedging Lab

In [ ]:
#read the parquet file
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date
from scipy.stats import norm
from scipy.interpolate import PchipInterpolator
from pathlib import Path


In [ ]:

# =============================================================================
# PART 1: BASIC MATHEMATICAL FUNCTIONS
# =============================================================================

def Phi(x):
    """Cumulative distribution function of the standard normal distribution."""
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def black_76(F, K, sigma, t, T, option_type='call'):
    """Black-76 formula for pricing European Call or Put options (interest rate r = 0)."""
    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * (T - t)) / (sigma * np.sqrt(T - t))
    d2 = d1 - sigma * np.sqrt(T - t)
    
    if option_type == 'call':
        return F * norm.cdf(d1) - K * norm.cdf(d2)
    elif option_type == 'put':
        return K * norm.cdf(-d2) - F * norm.cdf(-d1)
    else:
        raise ValueError("Invalid option type")

def vol_implicite(prix, F, K, T):
    """Extracts the implied volatility of an option using the bisection method."""
    bas, haut = 0.0001, 5.0
    for _ in range(100):  # Safety iteration limit to prevent infinite loops
        milieu = (bas + haut) / 2
        if black_76(F, K, milieu, 0, T, option_type='call') < prix:
            bas = milieu
        else:
            haut = milieu
        if haut - bas < 1e-6:
            break
    return (bas + haut) / 2

# =============================================================================
# PART 2: DATA LOADING AND DYNAMIC CORRELATION ESTIMATION
# =============================================================================

# 1. Load the clean parquet dataset


candidates = [Path.cwd() / 'unpivoted_bid_offer_train_clean.parquet', Path.cwd() / 'mes_donnees.csv']
path = next(p for p in candidates if p.exists())
df = pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
print("Data file:", path.name)
df['date'] = pd.to_datetime(df['date'])

valuation_date = pd.Timestamp('2020-01-06')
d = df[df['date'] == valuation_date]

c = d[d['underlying_ticker'] == 'COJ21 Comdty']   # April 2021 futures contract
c_K = d[d['underlying_ticker'] == 'COK21 Comdty'] # May 2021 futures contract

f1 = c['underlying_settlement'].iloc[0]
f2 = c_K['underlying_settlement'].iloc[0]


# 2. Dynamic estimation of the correlation rho up to the valuation date
def estimate_rho(data, recalibration_date):
    """Computes the historical correlation of log-returns between April and May futures."""
    tickers = ["COJ21 Comdty", "COK21 Comdty"]
    
    prices = (
        data.loc[data["date"] <= recalibration_date]
        [['date', 'underlying_ticker', 'underlying_settlement']]
        .drop_duplicates(subset=["date", "underlying_ticker"])
        .pivot(index="date", columns="underlying_ticker", values="underlying_settlement")
        .sort_index()
        [tickers]
        .dropna()
    )
    
    returns = np.log(prices / prices.shift(1)).dropna()
    
    # Fallback to a default high correlation if historical window is too short (e.g., initial valuation day)
    if len(returns) < 2:
        return 0.99  
        
    return returns["COJ21 Comdty"].corr(returns["COK21 Comdty"])

# Compute rho as of the valuation date
rho = estimate_rho(df, valuation_date)
print("Estimated correlation rho:", rho)
# =============================================================================
# PART 3: SMILE CALIBRATION (Polynomial vs PCHIP Interpolation)
# =============================================================================

# --- April contract (COJ21) smile extraction ---
strikes_1, vols_1 = [], []
for _, ligne in c.iterrows():
    if ligne['option_type'] == 'Call' and ligne['option_bid'] > 0:
        K = ligne['strike']
        prix = (ligne['option_bid'] + ligne['option_ask']) / 2
        F = ligne['underlying_settlement']
        T = ligne['time_to_expiry_days'] / 365
        strikes_1.append(K)
        vols_1.append(vol_implicite(prix, F, K, T))

coef_poly_1 = np.polyfit(strikes_1, vols_1, 3) # Degree-3 polynomial (for visual comparison)
smile_apr21 = PchipInterpolator(strikes_1, vols_1, extrapolate=True) # Robust PCHIP interpolator


# --- May contract (COK21) smile extraction ---
strikes_2, vols_2 = [], []
for _, ligne in c_K.iterrows():
    if ligne['option_type'] == 'Call' and ligne['option_bid'] > 0:
        K = ligne['strike']
        prix = (ligne['option_bid'] + ligne['option_ask']) / 2
        F = ligne['underlying_settlement']
        T = ligne['time_to_expiry_days'] / 365
        strikes_2.append(K)
        vols_2.append(vol_implicite(prix, F, K, T))

coef_poly_2 = np.polyfit(strikes_2, vols_2, 3)
smile_may21 = PchipInterpolator(strikes_2, vols_2, extrapolate=True)

# =============================================================================
# PART 4: ASIAN BASKET PRICING (MOMENT-MATCHING METHOD)
# =============================================================================

# 1. Time-to-fixing fractions (thetas) for each business day of February 2021
date_depart = valuation_date.date()
thetas = []
for jour in range(1, 29): 
    dt = date(2021, 2, jour)
    if dt.weekday() < 5:  # Business days only (Monday to Friday)
        thetas.append((dt - date_depart).days / 365.0)

n_total = len(thetas)  # Total 20 business days
n1 = 18                # 18 days allocated to April contract
n2 = n_total - n1      # 2 days allocated to May contract

theta_1 = thetas[:n1]  
theta_2 = thetas[n1:]  

# 2. Basket Forward price average (F_avg)
w1 = n1 / n_total
w2 = n2 / n_total
F_avg = w1 * f1 + w2 * f2

# 3. Strike projections (Sticky-Moneyness mapping)
K_call = 78.0
K1_call = f1 * (K_call / F_avg)
K2_call = f2 * (K_call / F_avg)

# 4. Read volatilities from PCHIP curves
sigma1 = float(smile_apr21(K1_call))
sigma2 = float(smile_may21(K2_call))

# 5. Moment-matching engine (Variances and Cross-Covariance)
def variance_totale(sigma1, sigma2, theta_1, theta_2, f1, f2, w1, w2, n1, n2):
    var_A1 = sum((2 * (n1 - k) + 1) * (math.exp(sigma1**2 * theta_1[k-1]) - 1) for k in range(1, n1 + 1))
    var_A1 *= (f1**2 / n1**2)

    var_A2 = sum((2 * (n2 - j) + 1) * (math.exp(sigma2**2 * theta_2[j-1]) - 1) for j in range(1, n2 + 1))
    var_A2 *= (f2**2 / n2**2)

    cov = sum(math.exp(rho * sigma1 * sigma2 * theta_1[k-1]) - 1 for k in range(1, n1 + 1))
    cov *= (f1 * f2 / n1)

    var_totale = (w1**2 * var_A1) + (w2**2 * var_A2) + (2 * w1 * w2 * cov)
    return var_totale

var_totale = variance_totale(sigma1, sigma2, theta_1, theta_2, f1, f2, w1, w2, n1, n2)
v = math.sqrt(math.log(1 + (var_totale / (F_avg**2))))

T_final = 1.0 
prix_call = black_76(F_avg, K_call, v, 0, T_final, option_type='call')

# =============================================================================
# PART 5: PUT STRIKE SEARCH (DICHOTOMY FOR ZERO-COST COLLAR)
# =============================================================================

bas = 40.0
haut = 78.0
prix_put = 0  

# Bisection loop to find K_put such that Put Price equals Call Price
for _ in range(100):
    K_put = (bas + haut) / 2
    K_put_1 = f1 * (K_put / F_avg)
    K_put_2 = f2 * (K_put / F_avg)
    
    sigma_put_1 = float(smile_apr21(K_put_1))
    sigma_put_2 = float(smile_may21(K_put_2))
    
    var_totale_put = variance_totale(sigma_put_1, sigma_put_2, theta_1, theta_2, f1, f2, w1, w2, n1, n2)
    v_put = math.sqrt(math.log(1 + (var_totale_put / (F_avg**2))))
    
    prix_put = black_76(F_avg, K_put, v_put, 0, T_final, option_type='put')
    
    if prix_put > prix_call:
        haut = K_put  # Put is too expensive -> lower the strike
    else:
        bas = K_put   # Put is too cheap -> raise the strike
        
    if abs(prix_put - prix_call) < 0.0001:
        break


# =============================================================================
# OUTPUTS AND COMPARATIVE PLOT (Polynomial vs PCHIP)
# =============================================================================
print("--- PRICING RESULTS (6 JANUARY 2020) ---")
print(f"Basket Forward price (F_avg): {F_avg:.2f} $")
print(f"Global basket volatility (v): {v:.4f}")
print(f"Theoretical Asian Call price at 78$: {prix_call:.4f} $")
print(f"Exact Put strike for Zero-Cost Collar: {K_put:.2f} $")
print(f"Verification - Call Price: {prix_call:.4f} $ | Put Price: {prix_put:.4f} $")

# Comparative plot: Polynomial degree 3 vs PCHIP
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
x_grid = np.linspace(min(strikes_1), max(strikes_1), 100)
plt.plot(strikes_1, vols_1, 'ko', label='Market Implied Vols')
plt.plot(x_grid, np.polyval(coef_poly_1, x_grid), 'b--', label='Degree-3 Polynomial')
plt.plot(x_grid, smile_apr21(x_grid), 'r-', label='PCHIP Interpolation')
plt.title('Smile Comparison: COJ21 (April)')
plt.xlabel('Strike')
plt.ylabel('Implied Volatility')
plt.legend()

plt.subplot(1, 2, 2)
x_grid_K = np.linspace(min(strikes_2), max(strikes_2), 100)
plt.plot(strikes_2, vols_2, 'ko', label='Market Implied Vols')
plt.plot(x_grid_K, np.polyval(coef_poly_2, x_grid_K), 'b--', label='Degree-3 Polynomial')
plt.plot(x_grid_K, smile_may21(x_grid_K), 'r-', label='PCHIP Interpolation')
plt.title('Smile Comparison: COK21 (May)')
plt.xlabel('Strike')
plt.ylabel('Implied Volatility')
plt.legend()

plt.tight_layout()
plt.show()

ICE Brent options are American but futures-style margined: the premium is not paid upfront but
marked to market daily, like the future itself. The holder therefore earns no interest by exercising early and receives the
same intrinsic value by keeping the option, so early exercise is never optimal and the American price equals the European one
(Lieu, 1990; Chen & Scott, 1993). This is why we invert the undiscounted Black-76 formula (r = 0, Asay 1982) and
treat the quotes as European.

In [ ]:
# =============================================================================
# PART 6: FIXING SCHEDULE FROM THE ICE EXPIRY RULE (ANNEX B)
# =============================================================================

MONTH_CODES = {'F': 1, 'G': 2, 'H': 3, 'J': 4, 'K': 5, 'M': 6,
               'N': 7, 'Q': 8, 'U': 9, 'V': 10, 'X': 11, 'Z': 12}

def futures_expiry(contract):
    """ICE Brent futures expire on the last business day of the second month preceding delivery."""
    code = contract.split()[0]                      # 'COJ21 Comdty' -> 'COJ21'
    delivery = pd.Timestamp(2000 + int(code[3:5]), MONTH_CODES[code[2]], 1)
    return delivery - pd.DateOffset(months=1) - pd.offsets.BDay(1)

def build_fixing_schedule(year, month, front_contract):
    """Business days of the averaging month and number of fixings on the front / next contract."""
    start = pd.Timestamp(year, month, 1)
    fixing_dates = pd.bdate_range(start, start + pd.offsets.MonthEnd(0))

    # The front contract fixes up to and including the second-to-last fixing day before its expiry
    before_expiry = fixing_dates[fixing_dates < futures_expiry(front_contract)]
    last_front_fixing = before_expiry[-2]

    n1 = int((fixing_dates <= last_front_fixing).sum())
    return fixing_dates, n1, len(fixing_dates) - n1

def fixing_thetas(fixing_dates, n1, day):
    """Year fractions from `day` to each fixing, split by leg."""
    thetas = ((fixing_dates - pd.Timestamp(day)).days / 365.0).to_numpy()
    return list(thetas[:n1]), list(thetas[n1:])


tickers = ['COJ21 Comdty', 'COK21 Comdty']
fixing_dates, n1_sched, n2_sched = build_fixing_schedule(2021, 2, tickers[0])

print("Apr21 futures expiry:", futures_expiry(tickers[0]).date())
print(f"Fixings: N = {len(fixing_dates)} | Apr21: {n1_sched} ({fixing_dates[0].date()} -> {fixing_dates[n1_sched-1].date()})"
      f" | May21: {n2_sched} ({fixing_dates[n1_sched].date()} -> {fixing_dates[-1].date()})")


theta_1_check, theta_2_check = fixing_thetas(fixing_dates, n1_sched, valuation_date)
assert (n1_sched, n2_sched) == (n1, n2)
assert np.allclose(theta_1_check, theta_1) and np.allclose(theta_2_check, theta_2)
print("Same schedule and thetas as the Main: OK")

In [ ]:
# =============================================================================
# PART 7: SMILE CALIBRATION WITH A LIQUIDITY FILTER
# =============================================================================

def get_smile(day_data, contract, max_rel_spread=0.5, min_mid=0.05):
    """Implied vols from liquid out-of-the-money quotes, interpolated with PCHIP."""
    quotes = day_data[day_data['underlying_ticker'] == contract]
    quotes = quotes[quotes[['option_bid', 'option_ask']].notna().all(axis=1) | quotes['settlement_price'].notna()]
    if quotes.empty:
        return None
    F = quotes['underlying_settlement'].iloc[0]

    strikes, vols = [], []
    for _, ligne in quotes.iterrows():
        bid, ask, K = ligne['option_bid'], ligne['option_ask'], ligne['strike']

        if pd.notna(bid) and pd.notna(ask):
            prix = (bid + ask) / 2
            # Liquidity filter
            if bid <= 0 or ask <= bid or prix < min_mid or (ask - bid) / prix > max_rel_spread:
                continue
        else:
            # From June 2020 the file only has settlement prices (settlement = mid when both exist)
            prix = ligne['settlement_price']
            if prix < min_mid:
                continue

        # Out-of-the-money only: calls above F, puts below F
        if ligne['option_type'] == 'Call' and K >= F:
            prix_call = prix
        elif ligne['option_type'] == 'Put' and K < F:
            prix_call = prix + F - K        # put-call parity with r = 0
        else:
            continue

        T = ligne['time_to_expiry_days'] / 365
        vol = vol_implicite(prix_call, F, K, T)
        if 0.01 < vol < 4.9:                # drop vols stuck on the bisection bounds
            strikes.append(K)
            vols.append(vol)

    if len(strikes) < 3:
        return None
    order = np.argsort(strikes)
    strikes, vols = np.array(strikes)[order], np.array(vols)[order]
    return {'F': F, 'strikes': strikes, 'vols': vols,
            'pchip': PchipInterpolator(strikes, vols, extrapolate=False)}

def read_vol(smile, K):
    """Smile volatility at strike K, flat outside the quoted range."""
    K = min(max(K, smile['strikes'][0]), smile['strikes'][-1])
    return float(smile['pchip'](K))


smile1_0 = get_smile(d, tickers[0])
smile2_0 = get_smile(d, tickers[1])
print(f"Apr21: {len(strikes_1)} quotes in the Main -> {len(smile1_0['strikes'])} liquid OTM quotes kept, "
      f"strikes {smile1_0['strikes'][0]:.0f} to {smile1_0['strikes'][-1]:.0f}")
print(f"May21: {len(strikes_2)} quotes in the Main -> {len(smile2_0['strikes'])} liquid OTM quotes kept, "
      f"strikes {smile2_0['strikes'][0]:.0f} to {smile2_0['strikes'][-1]:.0f}")

plt.figure(figsize=(14, 5))
for i, (smile, ks, vs, name) in enumerate([(smile1_0, strikes_1, vols_1, 'COJ21 (April)'),
                                            (smile2_0, strikes_2, vols_2, 'COK21 (May)')]):
    plt.subplot(1, 2, i + 1)
    grid = np.linspace(smile['strikes'][0], smile['strikes'][-1], 200)
    plt.plot(ks, vs, 'x', color='grey', label='Main: all calls')
    plt.plot(smile['strikes'], smile['vols'], 'ko', label='Filtered OTM quotes')
    plt.plot(grid, [read_vol(smile, k) for k in grid], 'r-', label='PCHIP')
    plt.axvline(smile['F'], color='k', ls=':', lw=0.8)
    plt.title(f'Smile {name} - 6 Jan 2020')
    plt.xlabel('Strike'); plt.ylabel('Implied Volatility'); plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# PART 8: PRICER AS FUNCTIONS (RHO AS AN ARGUMENT, SOFR DISCOUNTING)
# =============================================================================

def basket_variance(sigma1, sigma2, theta_1, theta_2, f1, f2, w1, w2, n1, n2, rho):
    """Same formulas as variance_totale, with rho passed explicitly."""
    var_A1 = sum((2 * (n1 - k) + 1) * (math.exp(sigma1**2 * theta_1[k-1]) - 1) for k in range(1, n1 + 1))
    var_A1 *= (f1**2 / n1**2)

    var_A2 = sum((2 * (n2 - j) + 1) * (math.exp(sigma2**2 * theta_2[j-1]) - 1) for j in range(1, n2 + 1))
    var_A2 *= (f2**2 / n2**2)

    cov = sum(math.exp(rho * sigma1 * sigma2 * theta_1[k-1]) - 1 for k in range(1, n1 + 1))
    cov *= (f1 * f2 / n1)

    return (w1**2 * var_A1) + (w2**2 * var_A2) + (2 * w1 * w2 * cov)

def price_asian(K, option_type, f1, f2, smile1, smile2, rho, theta_1, theta_2, D=1.0):
    """Moment-matching price of the Asian basket call/put, per barrel."""
    n1, n2 = len(theta_1), len(theta_2)
    w1, w2 = n1 / (n1 + n2), n2 / (n1 + n2)
    F_avg = w1 * f1 + w2 * f2

    sigma1 = read_vol(smile1, f1 * K / F_avg)
    sigma2 = read_vol(smile2, f2 * K / F_avg)

    var = basket_variance(sigma1, sigma2, theta_1, theta_2, f1, f2, w1, w2, n1, n2, rho)
    v = math.sqrt(math.log(1 + var / F_avg**2))
    return D * black_76(F_avg, K, v, 0, 1.0, option_type)

def solve_put_strike(call_price, f1, f2, smile1, smile2, rho, theta_1, theta_2, D=1.0, K_call=78.0):
    """Bisection on K_put such that Put(K_put) = Call(K_call)."""
    bas, haut = 1.0, K_call
    for _ in range(100):
        K_put = (bas + haut) / 2
        prix_put = price_asian(K_put, 'put', f1, f2, smile1, smile2, rho, theta_1, theta_2, D)
        if prix_put > call_price:
            haut = K_put
        else:
            bas = K_put
        if haut - bas < 1e-8:
            break
    return K_put

def discount_factor(day_data, day, payment_date):
    """SOFR discount factor (the dataset gives one overnight rate per day, in %)."""
    r = day_data['sofr_rate'].iloc[0] / 100
    return math.exp(-r * (pd.Timestamp(payment_date) - pd.Timestamp(day)).days / 365)


# Sanity check: with the Main's smiles and no discounting, the functions give back the Main's call
main_smile1 = {'strikes': np.array(strikes_1), 'pchip': smile_apr21}
main_smile2 = {'strikes': np.array(strikes_2), 'pchip': smile_may21}
check_call = price_asian(K_call, 'call', f1, f2, main_smile1, main_smile2, rho, theta_1, theta_2)
assert abs(check_call - prix_call) < 1e-10
print(f"price_asian reproduces the Main's call: {check_call:.6f} = {prix_call:.6f}")

In [ ]:
# =============================================================================
# PART 9: PRICING AS OF 6 JANUARY 2020 WITH THE NEW PIPELINE
# =============================================================================

payment_date = fixing_dates[-1]                  # assumption: the collar settles on the last fixing date
D_0 = discount_factor(d, valuation_date, payment_date)

call_0 = price_asian(K_call, 'call', f1, f2, smile1_0, smile2_0, rho, theta_1, theta_2, D_0)
K_put_0 = solve_put_strike(call_0, f1, f2, smile1_0, smile2_0, rho, theta_1, theta_2, D_0)
put_0 = price_asian(K_put_0, 'put', f1, f2, smile1_0, smile2_0, rho, theta_1, theta_2, D_0)

# Discounting does not move K_put: both legs share the same D
K_put_undiscounted = solve_put_strike(call_0 / D_0, f1, f2, smile1_0, smile2_0, rho, theta_1, theta_2)

notional = 1_000_000
print("--- PRICING RESULTS (6 JANUARY 2020) ---")
print(f"Discount factor to {payment_date.date()}: {D_0:.5f} (SOFR {d['sofr_rate'].iloc[0]:.2f}%)")
print(f"{'':28}{'Main':>12}{'New':>12}")
print(f"{'Call(78) $/bbl':28}{prix_call:12.4f}{call_0:12.4f}")
print(f"{'K_put $':28}{K_put:12.4f}{K_put_0:12.4f}")
print(f"{'Put(K_put) $/bbl':28}{prix_put:12.4f}{put_0:12.4f}")
print(f"{'Call premium on 1M bbl $':28}{prix_call * notional:12,.0f}{call_0 * notional:12,.0f}")
print(f"K_put without discounting: {K_put_undiscounted:.6f} (same as with discounting)")

In [ ]:
# =============================================================================
# PART 10: CALIBRATION ON EVERY DAY OF THE TRAINING SET
# =============================================================================

futures_prices = (
    df[['date', 'underlying_ticker', 'underlying_settlement']]
    .drop_duplicates(subset=['date', 'underlying_ticker'])
    .pivot(index='date', columns='underlying_ticker', values='underlying_settlement')
    .sort_index()
)
futures_prices.index = pd.to_datetime(futures_prices.index)
log_returns = np.log(futures_prices[tickers]).diff()

def estimate_rho_rolling(day, window=60, min_obs=20, default=0.99):
    """Correlation of daily log-returns over the last `window` days up to `day`."""
    r = log_returns.loc[:day].dropna().tail(window)
    if len(r) < min_obs:
        return default
    return r[tickers[0]].corr(r[tickers[1]])


calibration = []
last_smiles = {}

for day, day_data in df.groupby('date'):
    day = pd.Timestamp(day)
    if day > fixing_dates[-1]:
        break

    smiles, stale = {}, {}
    for contract in tickers:
        smile = get_smile(day_data, contract)
        stale[contract] = smile is None
        if smile is None:
            smile = last_smiles.get(contract)     # not enough liquid quotes: keep yesterday's smile
        last_smiles[contract] = smile
        smiles[contract] = smile

    f1_t, f2_t = futures_prices.at[day, tickers[0]], futures_prices.at[day, tickers[1]]
    rho_t = estimate_rho_rolling(day)
    row = {'date': day, 'f1': f1_t, 'f2': f2_t, 'rho': rho_t,
           'n_quotes_1': 0 if stale[tickers[0]] else len(smiles[tickers[0]]['strikes']),
           'n_quotes_2': 0 if stale[tickers[1]] else len(smiles[tickers[1]]['strikes']),
           'stale_1': stale[tickers[0]], 'stale_2': stale[tickers[1]],
           'has_bid_ask': day_data['option_bid'].notna().any()}

    if smiles[tickers[0]] is not None and np.isfinite(f1_t):
        row['atm_vol_1'] = read_vol(smiles[tickers[0]], f1_t)
    if smiles[tickers[1]] is not None:
        row['atm_vol_2'] = read_vol(smiles[tickers[1]], f2_t)

    # The moment-matching formulas assume no fixing has happened yet: during February 2021
    # the known fixings must be handled separately (Step 7)
    if day < fixing_dates[0]:
        D_t = discount_factor(day_data, day, payment_date)
        th1, th2 = fixing_thetas(fixing_dates, n1_sched, day)
        call_t = price_asian(K_call, 'call', f1_t, f2_t, smiles[tickers[0]], smiles[tickers[1]], rho_t, th1, th2, D_t)
        put_t = price_asian(K_put_0, 'put', f1_t, f2_t, smiles[tickers[0]], smiles[tickers[1]], rho_t, th1, th2, D_t)
        F_avg_t = (n1_sched * f1_t + n2_sched * f2_t) / (n1_sched + n2_sched)
        row.update({'D': D_t, 'F_avg': F_avg_t,
                    'sigma1_call': read_vol(smiles[tickers[0]], f1_t * K_call / F_avg_t),
                    'sigma1_put': read_vol(smiles[tickers[0]], f1_t * K_put_0 / F_avg_t),
                    'call': call_t, 'put': put_t,
                    'collar_value': notional * (put_t - call_t)})   # market maker: long put, short call
    calibration.append(row)

calibration = pd.DataFrame(calibration).set_index('date')
print(f"Calibrated days: {len(calibration)} ({calibration.index[0].date()} -> {calibration.index[-1].date()})")
print(f"Days with a stale smile: Apr21 {calibration['stale_1'].sum()}, May21 {calibration['stale_2'].sum()}")
print(f"Days priced from settlement prices only (no bid/ask): {(~calibration['has_bid_ask']).sum()}")
print(f"Collar value at inception: {calibration['collar_value'].iloc[0]:,.2f} $ (zero cost)")
display(calibration.drop(columns=['stale_1', 'stale_2', 'has_bid_ask']).iloc[::20].round(4))

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(14, 8))

ax[0, 0].plot(calibration[['f1', 'f2']])
ax[0, 0].axhline(K_call, color='r', ls='--', lw=0.8); ax[0, 0].axhline(K_put_0, color='g', ls='--', lw=0.8)
ax[0, 0].set_title('Futures prices (dashed: K_call and K_put)'); ax[0, 0].legend(['Apr21', 'May21'])

ax[0, 1].plot(calibration[['atm_vol_1', 'atm_vol_2', 'sigma1_put', 'sigma1_call']])
ax[0, 1].set_title('Implied volatilities'); ax[0, 1].legend(['ATM Apr21', 'ATM May21', 'Apr21 at put strike', 'Apr21 at call strike'])

ax[1, 0].plot(calibration['rho'])
ax[1, 0].set_title('Rolling 60-day correlation rho (0.99 before 20 observations)')

ax[1, 1].plot(calibration['collar_value'] / 1e6)
ax[1, 1].set_title('Collar value for the market maker (M$)')

plt.tight_layout()
plt.show()